# AMS-SkipGNN Kaggle runner

Upload **this** notebook: `notebooks/kaggle_runner.ipynb`.

Use **GPU T4**, Internet **ON**, then **Save Version → Save & Run All**.

Clones `aryonmt/finalProject` branch **`main`**. Override with `REPO_BRANCH`.

Default **`STAGE=5`** retrains GDI `gat` / `3hop` / `contrastive` with the **same per-batch encode** protocol as AMS. The earlier encode-once run (one Adam step per epoch) underfit: GAT hard AUPRC ~0.66 vs AMS ~0.83. Do not quote those extra-model rows.

| STAGE | What runs |
| --- | --- |
| `0` | Smoke: DTI `--quick` |
| `4` | Train `gat` / `3hop` / `contrastive` on DDI and PPI (skip if 3 seeds exist) |
| `5` (default) | Retrain those three models on **GDI only** (does **not** skip; overwrites encode-once rows) |

DTI / DDI / PPI extras are already in git. This run fills **GDI** `gat` / `3hop` / `contrastive`.

After the run, download **only** `/kaggle/working/ams_skipgnn_kaggle_bundle.zip` and drop it at the repo root (or send it back). Import folds CSVs, syncs `figures/`, and archives the executed notebook. Then the repo is delivery-ready.

GDI uses `--batch-size 1024` and encodes every decoder batch (same optimizer protocol as AMS, fewer steps than AMS batch 256). Skip-graph GAT is vectorized when it fits; otherwise chunked. Watch that GAT val AUPRC can keep rising toward the AMS ~0.95 range before you trust hard AUPRC.

**Resume a mid-run fair GDI extras job:** set env `SKIP_COMPLETE=1` and attach the previous Output. Do **not** set that flag if the attached CSVs are the encode-once underfit run.

Expected wall time on T4: GAT ~2 min/epoch at batch 1024 (not 15 min). Three seeds × 20 epochs is on the order of a few hours plus 3-hop and contrastive.


In [1]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

print('python', sys.version)
print('platform', platform.platform())
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as e:
    print('torch import failed', e)

REPO = 'https://github.com/aryonmt/finalProject.git'
BRANCH = os.environ.get('REPO_BRANCH', 'main')
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ROOT = WORK / 'finalProject'
if (Path.cwd() / 'src' / 'models').exists():
    ROOT = Path.cwd()
    print('already in repo', ROOT)
else:
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)])
    print('cloned', ROOT, 'branch', BRANCH)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('cwd', os.getcwd())


python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
platform Linux-6.12.90+-x86_64-with-glibc2.35


torch 2.10.0+cu128 cuda True
gpu Tesla T4


Cloning into '/kaggle/working/finalProject'...


cloned /kaggle/working/finalProject branch main
cwd /kaggle/working/finalProject


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'])
print('pip install -e . done')
subprocess.check_call([sys.executable, 'scripts/fetch_data.py'])
print('data fetch done')


pip install -e . done
+ git clone --depth 1 https://github.com/kexinhuang12345/SkipGNN.git /kaggle/working/finalProject/data/_upstream/SkipGNN


Cloning into '/kaggle/working/finalProject/data/_upstream/SkipGNN'...


copied DDI/train.csv
copied DDI/val.csv
copied DDI/test.csv
copied DDI/ddi_unique_smiles.csv
copied PPI/train.csv
copied PPI/val.csv
copied PPI/test.csv
copied PPI/protein_list.csv
copied DTI/train.csv
copied DTI/val.csv
copied DTI/test.csv
copied DTI/entity_list.csv
copied GDI/train.csv
copied GDI/val.csv
copied GDI/test.csv
copied GDI/entity_list.csv
DONE: data/raw is ready
data fetch done


In [3]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pytest', '-q'])
print('pytest rc', rc)
assert rc == 0, 'smoke tests failed'


....

..

.

.

.......

.                                                         [100%]


=============================== warnings summary ===============================
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDepr

pytest rc 0


In [4]:
import os, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd

stage = os.environ.get('STAGE', '5')
print('STAGE', stage, '(0=smoke, 4=DDI/PPI extras, 5=GDI extras)')
py = sys.executable
SEEDS = {7, 42, 123}
NEW_MODELS = ['gat', '3hop', 'contrastive']
if stage == '4':
    DATASETS = ['DDI', 'PPI']
    if os.environ.get('FILL_GDI_EXTRAS', '0') == '1':
        DATASETS.append('GDI')
else:
    DATASETS = ['GDI']


def _csv(ds):
    return Path(f'results/{ds}/benchmark.csv')


def _complete(ds, model):
    path = _csv(ds)
    if not path.exists():
        return False
    df = pd.read_csv(path)
    sub = df[df['model'].astype(str).str.lower() == model.lower()]
    if sub.empty or 'seed' not in sub.columns:
        return False
    have = set(int(s) for s in sub['seed'].dropna())
    if not SEEDS.issubset(have):
        return False
    if 'hard_auprc' in sub.columns:
        return bool(sub['hard_auprc'].notna().any())
    return True


BATCH = {'DDI': '1024', 'PPI': '1024', 'GDI': '1024'}


def _ingest_previous_kaggle_output():
    """Copy results/figures from a previous Kaggle version attached as input."""
    roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    copied = 0
    for root in roots:
        if not root.exists():
            continue
        for src in root.rglob('benchmark.csv'):
            ds = src.parent.name
            if ds not in {'DTI', 'DDI', 'PPI', 'GDI'}:
                continue
            if 'finalProject' in src.parts and src.resolve().is_relative_to(Path.cwd().resolve()):
                continue
            dest = Path('results') / ds
            dest.mkdir(parents=True, exist_ok=True)
            for item in src.parent.iterdir():
                target = dest / item.name
                if item.is_file():
                    if not target.exists() or item.stat().st_mtime >= target.stat().st_mtime:
                        shutil.copy2(item, target)
                        copied += 1
                elif item.is_dir() and item.name != '__pycache__':
                    shutil.copytree(item, target, dirs_exist_ok=True)
                    copied += 1
            print('ingested', src.parent, '->', dest, flush=True)
        for src in root.rglob('fig*.png'):
            if 'figures' not in src.parts:
                continue
            dest = Path('figures') / src.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists():
                shutil.copy2(src, dest)
                copied += 1
    print('ingest copied', copied, 'items', flush=True)


_ingest_previous_kaggle_output()


def _run(cmd):
    print('running', cmd, flush=True)
    subprocess.check_call(cmd)


t0 = time.time()
if stage == '0':
    _run([py, 'scripts/run_benchmark.py', '--dataset', 'DTI', '--models', 'gcn', '--quick'])
else:
    print('=== extras: gat / 3hop / contrastive on', DATASETS, '===')
    failed = []
    for ds in DATASETS:
        for m in NEW_MODELS:
            skip_done = stage == '4' or os.environ.get('SKIP_COMPLETE', '0') == '1'
            if skip_done and _complete(ds, m):
                print(f'[{ds}] {m} already complete, skip')
                continue
            cmd = [
                py, 'scripts/run_benchmark.py',
                '--dataset', ds,
                '--models', m,
                '--batch-size', BATCH[ds],
                '--save-embeddings',
            ]
            try:
                _run(cmd)
            except subprocess.CalledProcessError as exc:
                print(f'FAILED {ds} {m} rc={exc.returncode}', flush=True)
                failed.append((ds, m, exc.returncode))
    print('training cell done in', round((time.time() - t0) / 60, 2), 'min')
    if failed:
        print('FAILED RUNS (continuing to pack whatever finished):', failed, flush=True)
    for ds in ['DTI', *DATASETS]:
        path = _csv(ds)
        if not path.exists():
            print(f'[{ds}] missing benchmark.csv')
            continue
        df = pd.read_csv(path)
        print(ds, sorted(df['model'].astype(str).str.lower().unique().tolist()))


STAGE 5 (0=smoke, 4=DDI/PPI extras, 5=GDI extras)
ingest copied 0 items


=== extras: gat / 3hop / contrastive on ['GDI'] ===
running ['/usr/bin/python3', 'scripts/run_benchmark.py', '--dataset', 'GDI', '--models', 'gat', '--batch-size', '1024', '--save-embeddings']


dataset=GDI device=cuda epochs=20 batch_size=1024 seeds=[42, 123, 7] models=['gat'] encode_once=False


loaded GDI: n=19783 src=9413 tgt=10370 train=114445 val=16349 test=32698
=== gat seed=42 ===


epoch=01 loss=0.3189 val_auprc=0.9108


...

epoch=09 loss=0.5758 val_auprc=0.9475


epoch=10 loss=0.5670 val_auprc=0.9475


contrastive seed=7 uniform_auprc=0.9495 hard_auprc=0.8240


saved embeddings (19783, 64)
dataset       model  seed  uniform_auprc  hard_auprc  uniform_auroc  hard_auroc   f1_tau  tau_star  best_val_auprc              method
    GDI         gcn     7       0.934700    0.757664       0.922893    0.745238 0.862981      0.38        0.933550                 NaN
    GDI         gcn    42       0.933430    0.743867       0.921032    0.726038 0.859617      0.45        0.932826                 NaN
    GDI         gcn   123       0.934200    0.744569       0.925055    0.725144 0.857649      0.46        0.933449                 NaN
    GDI     skipgnn     7       0.934523    0.725946       0.924853    0.710416 0.858116      0.31        0.933426                 NaN
    GDI     skipgnn    42       0.935133    0.728095       0.925758    0.710485 0.859570      0.32        0.933767                 NaN
    GDI     skipgnn   123       0.931029    0.705517       0.917705    0.693476 0.860014      0.48        0.928850                 NaN
    GDI         ams     7 

training cell done in 279.97 min
DTI ['3hop', 'ams', 'contrastive', 'gat', 'gcn', 'heuristic', 'skipgnn']
GDI ['3hop', 'ams', 'contrastive', 'gat', 'gcn', 'heuristic', 'skipgnn']


In [5]:
import json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path

subprocess.call([sys.executable, 'scripts/plot_tsne.py'])
subprocess.call([sys.executable, 'scripts/make_figures.py', '--skip-tsne'])

repo = Path.cwd()
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else repo
staging = work / '_kaggle_bundle_staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)


def copy_tree(src: Path, dest: Path) -> int:
    if not src.exists():
        return 0
    n = 0
    dest.mkdir(parents=True, exist_ok=True)
    for path in src.rglob('*'):
        if not path.is_file() or path.name in {'.gitkeep', '.DS_Store'}:
            continue
        if path.suffix.lower() in {'.pt', '.pth', '.ckpt', '.zip'}:
            continue
        if 'checkpoints' in path.parts or path.parts[:1] == ('temp',) or 'temp' in path.parts:
            continue
        target = dest / path.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
        n += 1
    return n


n_results = copy_tree(repo / 'results', staging / 'results')
n_figures = copy_tree(repo / 'figures', staging / 'figures')

nb_candidates = [
    Path('/kaggle/working/__notebook__.ipynb'),
    Path('/kaggle/working/__notebook_source__.ipynb'),
    work / 'kaggle_runner.ipynb',
    repo / 'notebooks' / 'kaggle_runner.ipynb',
    work / 'kaggle_runner.ipynb',
    repo / 'notebooks' / 'kaggle_runner.ipynb',
]
nb_src = next((p for p in nb_candidates if p.is_file()), None)
if nb_src is not None:
    dest_nb = staging / 'notebooks' / 'kaggle_runner.ipynb'
    dest_nb.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(nb_src, dest_nb)

files = sorted(p.relative_to(staging).as_posix() for p in staging.rglob('*') if p.is_file())
manifest = {
    'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'stage': os.environ.get('STAGE', '5'),
    'cwd': str(repo),
    'n_result_files': n_results,
    'n_figure_files': n_figures,
    'notebook_source': str(nb_src) if nb_src else None,
    'files': files,
    'import_map': {
        'results/': 'results/',
        'figures/': 'figures/',
        'notebooks/kaggle_runner.ipynb': 'notebooks/executed/kaggle_run.ipynb',
    },
}
try:
    import torch
    manifest['torch'] = torch.__version__
    manifest['cuda'] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        manifest['gpu'] = torch.cuda.get_device_name(0)
except Exception:
    pass
(staging / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(staging / 'IMPORT.txt').write_text(
    'Drop this zip at the repo root and tell the assistant.\n'
    'It will merge results/, sync figures/, archive the executed notebook, and leave the repo delivery-ready.\n'
    'Command: python scripts/import_kaggle_bundle.py --src ams_skipgnn_kaggle_bundle.zip '
    '--archive-notebook notebooks/executed/kaggle_run.ipynb --make-figures\n',
    encoding='utf-8',
)

zip_path = work / 'ams_skipgnn_kaggle_bundle.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in staging.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(staging).as_posix())
shutil.rmtree(staging, ignore_errors=True)

print('DOWNLOAD THIS FILE:', zip_path)
print('bytes', zip_path.stat().st_size)
print('files', len(files))
print('KAGGLE GDI EXTRAS COMPLETE — download only ams_skipgnn_kaggle_bundle.zip')


t-SNE figures written to /kaggle/working/finalProject/figures
